# **Machine Translation**

In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()
df=pd.read_csv("rus.txt", sep="\t", header=None, usecols=[0, 1], names=["english", "russian"])
print("first 5 rows")
print(df.head())
print("total sentance pairs")
print(len(df))

Saving rus.txt to rus.txt
first 5 rows
  english        russian
0     Go.          Марш!
1     Go.           Иди.
2     Go.         Идите.
3     Hi.  Здравствуйте.
4     Hi.        Привет!
total sentance pairs
363386


**Data preprocessing**

In [ ]:
import re
import string
from sklearn.model_selection import train_test_split

file = "rus.txt"
pairs = []

with open(file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            eng = parts[0]
            rus = parts[1]
            pairs.append((eng, rus))

print(pairs[:10])
print(len(pairs))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\d+", "", text)  # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

cleaned_pairs = []
for eng, rus in pairs:
    eng = clean_text(eng)
    rus = clean_text(rus)
    cleaned_pairs.append((eng, rus))

print(cleaned_pairs[:10])

[('Go.', 'Марш!'), ('Go.', 'Иди.'), ('Go.', 'Идите.'), ('Hi.', 'Здравствуйте.'), ('Hi.', 'Привет!'), ('Hi.', 'Хай.'), ('Hi.', 'Здрасте.'), ('Hi.', 'Здоро́во!'), ('Run!', 'Беги!'), ('Run!', 'Бегите!')]
363386
[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здоро́во'), ('run', 'беги'), ('run', 'бегите')]


In [ ]:
filtered_pairs = []

for eng, rus in cleaned_pairs:
    if len(eng.split()) <= 10 and len(rus.split()) <= 10:
        filtered_pairs.append((eng, rus))

print("Filtered pairs:", len(filtered_pairs))
print(filtered_pairs[:10])

Filtered pairs: 348679
[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здоро́во'), ('run', 'беги'), ('run', 'бегите')]


In [ ]:
processed_pairs = []

for eng, rus in filtered_pairs:
    rus = "<sos> " + rus + " <eos>"
    processed_pairs.append((eng, rus))

print(processed_pairs[:10])

[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>'), ('hi', '<sos> хай <eos>'), ('hi', '<sos> здрасте <eos>'), ('hi', '<sos> здоро́во <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>')]


In [ ]:
MAX_SAMPLES = 100000
processed_pairs = processed_pairs[:MAX_SAMPLES]

print("Using pairs:", len(processed_pairs))
print(processed_pairs[:5])

Using pairs: 100000
[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>')]


In [ ]:
from collections import Counter

def build_vocab(sentences, special_tokens, max_vocab_size=None):
    counter = Counter()

    for sentence in sentences:
        words = sentence.split()
        counter.update(words)

    vocab = {}
    for token in special_tokens:
        vocab[token] = len(vocab)

    if max_vocab_size is None:
        words_to_add = counter.items()
    else:
        words_to_add = counter.most_common(max_vocab_size)

    for item in words_to_add:
        word = item[0]
        if word not in vocab:
            vocab[word] = len(vocab)

    return vocab

In [ ]:
english_sentences = []
russian_sentences = []

for eng, rus in processed_pairs:
    english_sentences.append(eng)
    russian_sentences.append(rus)

In [ ]:
ENG_VOCAB_SIZE = 12000
RUS_VOCAB_SIZE = 16000

eng_vocab = build_vocab(
    english_sentences,
    ["<pad>", "<unk>"],
    max_vocab_size=ENG_VOCAB_SIZE
)

rus_vocab = build_vocab(
    russian_sentences,
    ["<pad>", "<unk>", "<sos>", "<eos>"],
    max_vocab_size=RUS_VOCAB_SIZE
)

print("English vocab size:", len(eng_vocab))
print("Russian vocab size:", len(rus_vocab))
print(list(eng_vocab.items())[:10])
print(list(rus_vocab.items())[:10])

English vocab size: 7218
Russian vocab size: 16002
[('<pad>', 0), ('<unk>', 1), ('tom', 2), ('i', 3), ('you', 4), ('is', 5), ('a', 6), ('it', 7), ('to', 8), ('the', 9)]
[('<pad>', 0), ('<unk>', 1), ('<sos>', 2), ('<eos>', 3), ('я', 4), ('том', 5), ('не', 6), ('это', 7), ('ты', 8), ('вы', 9)]


In [ ]:
def sentence_to_indices(sentence, vocab):
    if isinstance(sentence, str):
        sentence = sentence.split()

    return [vocab.get(word, vocab["<unk>"]) for word in sentence]

In [ ]:
data = []

for eng, rus in processed_pairs:
    eng_indices = sentence_to_indices(eng, eng_vocab)
    rus_indices = sentence_to_indices(rus, rus_vocab)
    data.append((eng_indices, rus_indices))

print(data[:10])

[([25], [2, 5642, 3]), ([25], [2, 223, 3]), ([25], [2, 331, 3]), ([982], [2, 4881, 3]), ([982], [2, 866, 3]), ([982], [2, 12115, 3]), ([982], [2, 12116, 3]), ([982], [2, 12117, 3]), ([331], [2, 3491, 3]), ([331], [2, 4882, 3])]


In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

print("Sample train:", train_data[0])
print("Sample val:", val_data[0])

Train size: 80000
Validation size: 20000
Sample train: ([3, 22, 6, 87, 1658], [2, 12, 15, 290, 2375, 3])
Sample val: ([3, 31, 20, 5009], [2, 10, 58, 346, 1, 3])


**Create data loaders**

In [ ]:
train_src = []
train_tgt = []

for eng_indices, rus_indices in train_data:
    train_src.append(eng_indices)
    train_tgt.append(rus_indices)

val_src = []
val_tgt = []

for eng_indices, rus_indices in val_data:
    val_src.append(eng_indices)
    val_tgt.append(rus_indices)

print("Train examples:", len(train_src))
print("Validation examples:", len(val_src))

Train examples: 80000
Validation examples: 20000


In [ ]:
train_src_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_src,
    padding="post",
    value=eng_vocab["<pad>"]
)

train_tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_tgt,
    padding="post",
    value=rus_vocab["<pad>"]
)

val_src_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_src,
    padding="post",
    value=eng_vocab["<pad>"]
)

val_tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_tgt,
    padding="post",
    value=rus_vocab["<pad>"]
)

print("Train source shape:", train_src_padded.shape)
print("Train target shape:", train_tgt_padded.shape)
print("Validation source shape:", val_src_padded.shape)
print("Validation target shape:", val_tgt_padded.shape)

Train source shape: (80000, 7)
Train target shape: (80000, 12)
Validation source shape: (20000, 7)
Validation target shape: (20000, 12)


In [ ]:
# =========================================
# HYPERPARAMETERS
# =========================================
INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(rus_vocab)
EMB_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 1
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10

PAD_IDX = rus_vocab["<pad>"]
SOS_IDX = rus_vocab["<sos>"]
EOS_IDX = rus_vocab["<eos>"]

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_src_padded, train_tgt_padded))
train_dataset = train_dataset.shuffle(len(train_src_padded)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_src_padded, val_tgt_padded))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for src_batch, tgt_batch in train_dataset.take(1):
    print("Source batch shape:", src_batch.shape)
    print("Target batch shape:", tgt_batch.shape)

Source batch shape: (32, 7)
Target batch shape: (32, 12)


**Model architecture**

In [ ]:
# =========================================
# MODEL ARCHITECTURE
# =========================================
import tensorflow as tf

class Encoder(tf.keras.Model):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(input_dim, emb_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(
            hidden_dim,
            return_sequences=True,
            return_state=True
        )

    def call(self, x):
        x = self.embedding(x)
        outputs, hidden_state, cell_state = self.lstm(x)
        return outputs, hidden_state, cell_state


class Decoder(tf.keras.Model):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(output_dim, emb_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(
            hidden_dim,
            return_sequences=True,
            return_state=True
        )
        self.fc = tf.keras.layers.Dense(output_dim)

    def call(self, x, hidden_state, cell_state):
        x = self.embedding(x)
        outputs, hidden_state, cell_state = self.lstm(
            x, initial_state=[hidden_state, cell_state]
        )
        predictions = self.fc(outputs)
        return predictions, hidden_state, cell_state


class Seq2Seq(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, src, tgt, training=False, teacher_forcing_ratio=0.5):
        tgt_len = tgt.shape[1]

        _, hidden_state, cell_state = self.encoder(src)

        decoder_input = tf.expand_dims(tgt[:, 0], 1)   # <sos>
        outputs = []

        for t in range(1, tgt_len):
            predictions, hidden_state, cell_state = self.decoder(
                decoder_input, hidden_state, cell_state
            )

            outputs.append(predictions[:, 0, :])

            predicted_token = tf.argmax(
                predictions[:, 0, :], axis=-1, output_type=tf.int32
            )

            if training and tf.random.uniform(()) < teacher_forcing_ratio:
                next_input = tgt[:, t]
            else:
                next_input = predicted_token

            decoder_input = tf.expand_dims(next_input, 1)

        outputs = tf.stack(outputs, axis=1)   # (batch, tgt_len-1, vocab_size)
        return outputs

In [ ]:
encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder)

print(model)

<Seq2Seq name=seq2_seq, built=False>


In [ ]:
for src_batch, tgt_batch in train_dataset.take(1):
    outputs = model(src_batch, tgt_batch)
    print("Output shape:", outputs.shape)
    break

Output shape: (32, 11, 16002)


**Training**

In [20]:
# =========================================
# TRAINING
# =========================================
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def masked_loss(y_true, y_pred):
    loss = loss_fn(y_true, y_pred)   # (batch, tgt_len-1)

    mask = tf.cast(tf.not_equal(y_true, PAD_IDX), tf.float32)
    loss = loss * mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def train_step(src, tgt):
    with tf.GradientTape() as tape:
        output = model(src, tgt, training=True, teacher_forcing_ratio=0.5)
        real = tgt[:, 1:]   # remove <sos>
        loss = masked_loss(real, output)

    variables = model.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return loss

def evaluate_step(src, tgt):
    output = model(src, tgt, training=False, teacher_forcing_ratio=0.0)
    real = tgt[:, 1:]   # remove <sos>
    loss = masked_loss(real, output)

    return loss

def train():
    total_loss = 0.0
    num_batches = 0

    for src_batch, tgt_batch in train_dataset:
        loss = train_step(src_batch, tgt_batch)
        total_loss += float(loss.numpy())
        num_batches += 1

        if num_batches % 500 == 0:
            print(f"Train Batch {num_batches}: loss = {total_loss / num_batches:.4f}")

    return total_loss / num_batches

def evaluate():
    total_loss = 0.0
    num_batches = 0

    for src_batch, tgt_batch in val_dataset:
        loss = evaluate_step(src_batch, tgt_batch)
        total_loss += float(loss.numpy())
        num_batches += 1

    return total_loss / num_batches

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train()
    val_loss = evaluate()

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print("-" * 30)


Epoch 1/10
Train Batch 500: loss = 5.3729
Train Batch 1000: loss = 4.9541
Train Batch 1500: loss = 4.6381
Train Batch 2000: loss = 4.3860
Train Batch 2500: loss = 4.1673
Train Loss: 4.1673
Val Loss:   3.2649
------------------------------

Epoch 2/10
Train Batch 500: loss = 2.7480
Train Batch 1000: loss = 2.6659
Train Batch 1500: loss = 2.5833
Train Batch 2000: loss = 2.5082
Train Batch 2500: loss = 2.4342
Train Loss: 2.4342
Val Loss:   2.4339
------------------------------

Epoch 3/10
Train Batch 500: loss = 1.6346
Train Batch 1000: loss = 1.6311
Train Batch 1500: loss = 1.6234
Train Batch 2000: loss = 1.6056
Train Batch 2500: loss = 1.5903
Train Loss: 1.5903
Val Loss:   2.1256
------------------------------

Epoch 4/10
Train Batch 500: loss = 1.1018
Train Batch 1000: loss = 1.1341
Train Batch 1500: loss = 1.1492
Train Batch 2000: loss = 1.1633
Train Batch 2500: loss = 1.1726
Train Loss: 1.1726
Val Loss:   2.0539
------------------------------

Epoch 5/10
Train Batch 500: loss = 0.85

In [21]:
# =========================================
# TRANSLATION FUNCTION
# =========================================
def translate_sentence(sentence):
    sentence = clean_text(sentence)
    words = sentence.split()

    src_indices = []
    for word in words:
        src_indices.append(eng_vocab.get(word, eng_vocab["<unk>"]))

    src_tensor = tf.keras.preprocessing.sequence.pad_sequences(
        [src_indices],
        padding="post"
    )
    src_tensor = tf.convert_to_tensor(src_tensor, dtype=tf.int32)

    encoder_outputs, hidden_state, cell_state = model.encoder(src_tensor)

    decoder_input = tf.constant([[SOS_IDX]], dtype=tf.int32)

    result = []

    for _ in range(20):
        predictions, hidden_state, cell_state = model.decoder(
            decoder_input, hidden_state, cell_state
        )

        predicted_id = int(tf.argmax(predictions[0, 0]).numpy())

        if predicted_id == EOS_IDX:
            break

        predicted_word = None
        for word, idx in rus_vocab.items():
            if idx == predicted_id:
                predicted_word = word
                break

        if predicted_word is None:
            predicted_word = "<unk>"

        if predicted_word not in ["<sos>", "<eos>", "<pad>"]:
            result.append(predicted_word)

        decoder_input = tf.constant([[predicted_id]], dtype=tf.int32)

    return " ".join(result)

In [22]:
print(translate_sentence("i am tired"))
print(translate_sentence("he is my friend"))
print(translate_sentence("we love music"))
print(translate_sentence("go"))
print(translate_sentence("hi"))
print(translate_sentence("run"))

я устала
он мой друг
мы любим музыку
иди
привет
беги
